# Data Cleaning

This notebook creates reproducible outputs without modifying the raw dataset. The cleaning rationale is documented in `../../docs/milestone_04_km_driven_cleaning.md`. 

In [1]:
from pathlib import Path

import pandas as pd

RAW_PATH = Path("../data/raw/cardekho_dataset.csv")
OUTPUT_PATH = Path("../data/processed/cardekho_model_ready.csv")
MANIFEST_PATH = Path("../data/processed/cleaning_exclusions_manifest.csv")
TARGET = "selling_price"
DROP_COLUMNS = ["Unnamed: 0", "car_name"]
EXPECTED_COLUMNS = [
    "brand", "model", "vehicle_age", "km_driven", "seller_type",
    "fuel_type", "transmission_type", "mileage", "engine",
    "max_power", "seats", TARGET,
]

raw_df = pd.read_csv(RAW_PATH)
assert raw_df.shape == (15_411, 14)
raw_df.shape

(15411, 14)

## Raw-data audit

These checks are deliberately performed before cleaning. They establish the data-quality baseline and do not mutate `raw_df`.

In [2]:
missing_values = raw_df.isna().sum().rename("missing_values")
data_types = raw_df.dtypes.astype(str).rename("dtype")
raw_duplicate_count = int(raw_df.duplicated().sum())
rows_without_identifiers = raw_df.drop(columns=DROP_COLUMNS)
duplicate_count_after_identifier_removal = int(rows_without_identifiers.duplicated().sum())

invalid_numeric_counts = pd.Series({
    "vehicle_age < 0": int((raw_df["vehicle_age"] < 0).sum()),
    "vehicle_age == 0": int((raw_df["vehicle_age"] == 0).sum()),
    "km_driven < 0": int((raw_df["km_driven"] < 0).sum()),
    "mileage <= 0": int((raw_df["mileage"] <= 0).sum()),
    "engine <= 0": int((raw_df["engine"] <= 0).sum()),
    "max_power <= 0": int((raw_df["max_power"] <= 0).sum()),
    "seats <= 0": int((raw_df["seats"] <= 0).sum()),
    "selling_price <= 0": int((raw_df[TARGET] <= 0).sum()),
}, name="count")

assert raw_duplicate_count == 0
assert duplicate_count_after_identifier_removal == 167
assert invalid_numeric_counts["vehicle_age < 0"] == 0
assert invalid_numeric_counts["seats <= 0"] == 2

raw_audit_summary = pd.concat([missing_values, data_types], axis=1)
print(f"Full-row duplicates in raw data: {raw_duplicate_count}")
print(f"Duplicates after removing identifiers: {duplicate_count_after_identifier_removal}")
display(raw_audit_summary)
display(invalid_numeric_counts.to_frame())

Full-row duplicates in raw data: 0
Duplicates after removing identifiers: 167


,missing_values,dtype
Unnamed: 0,0,int64
car_name,0,str
brand,0,str
model,0,str
vehicle_age,0,int64
km_driven,0,int64
seller_type,0,str
fuel_type,0,str
transmission_type,0,str
mileage,0,float64


,count
vehicle_age < 0,0
vehicle_age == 0,5
km_driven < 0,0
mileage <= 0,0
engine <= 0,0
max_power <= 0,0
seats <= 0,2
selling_price <= 0,0


## Targeted `km_driven` label-quality audit

High mileage alone is not an exclusion criterion. The `km_driven == selling_price` condition is a one-time label-quality audit used to identify corrupted training rows; it is not a prediction feature and is not retained in the output. Annual mileage is calculated only for positive `vehicle_age`; age-zero vehicles are treated as potentially valid same-year vehicles and are retained.

In [3]:
assert (raw_df["vehicle_age"] < 0).sum() == 0
annual_km = raw_df["km_driven"].div(raw_df["vehicle_age"].where(raw_df["vehicle_age"] > 0))
copied_price_as_km = (raw_df["km_driven"] > 500_000) & (raw_df["km_driven"] == raw_df[TARGET])
implausible_annual_km = annual_km > 250_000
exclude_for_corrupt_km = copied_price_as_km | implausible_annual_km

km_exclusion_reason = pd.Series("", index=raw_df.index, dtype="string")
km_exclusion_reason.loc[copied_price_as_km] = "km_driven_matches_selling_price_above_500k"
km_exclusion_reason.loc[implausible_annual_km] = "annual_km_exceeds_250000"

km_audit = raw_df.loc[exclude_for_corrupt_km, ["Unnamed: 0", "car_name", "vehicle_age", "km_driven", TARGET]].copy()
km_audit = km_audit.assign(
    km_per_year=annual_km.loc[exclude_for_corrupt_km],
    exclusion_reason=km_exclusion_reason.loc[exclude_for_corrupt_km],
).rename(columns={"Unnamed: 0": "raw_row_id"})

assert len(km_audit) == 7
assert (raw_df["vehicle_age"] == 0).sum() == 5
km_audit.sort_values("raw_row_id")

,raw_row_id,car_name,vehicle_age,km_driven,selling_price,km_per_year,exclusion_reason
4532,5754,Renault Duster,4,850000,850000,212500.000000,km_driven_matches_selling_price_above_500k
7534,9636,Toyota Fortuner,2,590000,2850000,295000.000000,annual_km_exceeds_250000
8772,11291,Toyota Innova,6,950000,950000,158333.333333,km_driven_matches_selling_price_above_500k
9170,11792,Skoda Rapid,5,675000,675000,135000.000000,km_driven_matches_selling_price_above_500k
10384,13242,Honda Jazz,4,525000,525000,131250.000000,km_driven_matches_selling_price_above_500k
10838,13790,Hyundai Creta,6,825000,825000,137500.000000,km_driven_matches_selling_price_above_500k
15409,19542,Mahindra XUV500,5,3800000,1225000,760000.000000,annual_km_exceeds_250000


## Build the modelling table and exclusion manifest

After the seven corrupted-km rows are removed, `Unnamed: 0` and `car_name` are dropped. Exact duplicates are then removed from the resulting modelling columns (keeping the first occurrence). Finally, the two zero-seat rows are dropped because zero is not a valid vehicle seat count.

In [4]:
after_km_df = raw_df.loc[~exclude_for_corrupt_km].copy()
modelling_candidates = after_km_df.drop(columns=DROP_COLUMNS)
duplicate_modelling_row = modelling_candidates.duplicated(keep="first")
assert int(duplicate_modelling_row.sum()) == 167

duplicate_manifest = after_km_df.loc[duplicate_modelling_row, ["Unnamed: 0"]].rename(columns={"Unnamed: 0": "raw_row_id"})
duplicate_manifest["exclusion_reason"] = "exact_duplicate_after_identifier_removal"

after_dedup_df = modelling_candidates.loc[~duplicate_modelling_row].copy()
invalid_seats = after_dedup_df["seats"] <= 0
assert int(invalid_seats.sum()) == 2
seat_manifest = after_km_df.loc[~duplicate_modelling_row & (after_km_df["seats"] <= 0), ["Unnamed: 0"]].rename(columns={"Unnamed: 0": "raw_row_id"})
seat_manifest["exclusion_reason"] = "invalid_seats_less_than_or_equal_to_zero"

model_df = after_dedup_df.loc[~invalid_seats].reset_index(drop=True)
manifest = pd.concat([
    km_audit[["raw_row_id", "exclusion_reason"]],
    duplicate_manifest,
    seat_manifest,
], ignore_index=True).sort_values("raw_row_id").reset_index(drop=True)

assert len(manifest) == 176
assert manifest["raw_row_id"].is_unique
assert manifest["exclusion_reason"].value_counts().to_dict() == {
    "exact_duplicate_after_identifier_removal": 167,
    "km_driven_matches_selling_price_above_500k": 5,
    "annual_km_exceeds_250000": 2,
    "invalid_seats_less_than_or_equal_to_zero": 2,
}
manifest.head(10)

,raw_row_id,exclusion_reason
0,261,exact_duplicate_after_identifier_removal
1,462,exact_duplicate_after_identifier_removal
2,1739,exact_duplicate_after_identifier_removal
3,1838,exact_duplicate_after_identifier_removal
4,1907,exact_duplicate_after_identifier_removal
5,2318,exact_duplicate_after_identifier_removal
6,2507,exact_duplicate_after_identifier_removal
7,2512,exact_duplicate_after_identifier_removal
8,3308,exact_duplicate_after_identifier_removal
9,4090,exact_duplicate_after_identifier_removal


## Validate and save derived outputs

The final assertions guard the agreed schema and all completed Milestone 4 decisions.

In [5]:
assert model_df.columns.tolist() == EXPECTED_COLUMNS
assert not model_df.columns.str.startswith("Unnamed").any()
assert model_df.duplicated().sum() == 0
assert (model_df["seats"] <= 0).sum() == 0
assert len(model_df) == 15_235

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
model_df.to_csv(OUTPUT_PATH, index=False)
manifest.to_csv(MANIFEST_PATH, index=False)

final_audit_summary = pd.Series({
    "raw_rows": len(raw_df),
    "km_corruption_rows_removed": len(km_audit),
    "duplicate_modelling_rows_removed": int(duplicate_modelling_row.sum()),
    "invalid_seat_rows_removed": int(invalid_seats.sum()),
    "final_rows": len(model_df),
    "final_columns": len(model_df.columns),
    "final_exact_duplicate_rows": int(model_df.duplicated().sum()),
    "final_invalid_seat_rows": int((model_df["seats"] <= 0).sum()),
}, name="count")
display(final_audit_summary.to_frame())
print(f"Wrote {len(model_df):,} rows to {OUTPUT_PATH}")
print(f"Wrote {len(manifest):,} exclusion records to {MANIFEST_PATH}")

,count
raw_rows,15411
km_corruption_rows_removed,7
duplicate_modelling_rows_removed,167
invalid_seat_rows_removed,2
final_rows,15235
final_columns,12
final_exact_duplicate_rows,0
final_invalid_seat_rows,0


Wrote 15,235 rows to ..\data\processed\cardekho_model_ready.csv
Wrote 176 exclusion records to ..\data\processed\cleaning_exclusions_manifest.csv
